# 03 — Machine Learning: Predicting Business Profitability

This notebook develops regression and classification models to predict business profitability using the Superstore dataset.

The objective is to identify the factors influencing profitability, predict transaction-level profit, and classify transactions according to their risk of generating a loss.

In [1]:

# ============================================
# STEP 1: IMPORT LIBRARIES AND LOAD THE DATA
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Check machine learning library
print("Scikit-learn version:", sklearn.__version__)

# Locate the dataset
possible_paths = [
    Path.cwd().parent / "data" / "Superstore.xls",
    Path.cwd() / "data" / "Superstore.xls",
]

file_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if file_path is None:
    raise FileNotFoundError(
        "Superstore.xls was not found. Check the data folder."
    )

print("\nDataset found:", file_path)

# Load the three sheets
orders = pd.read_excel(file_path, sheet_name="Orders")
returns = pd.read_excel(file_path, sheet_name="Returns")
people = pd.read_excel(file_path, sheet_name="People")

# Display dataset dimensions
print("\nDATASET DIMENSIONS")
print("Orders:", orders.shape)
print("Returns:", returns.shape)
print("People:", people.shape)

# Verify required columns
print("\nORDERS COLUMNS:")
print(orders.columns.tolist())

# Display first five observations
display(orders.head())


Scikit-learn version: 1.9.1

Dataset found: c:\Users\USER\Desktop\Supersore_Profitability_Project\data\Superstore.xls


C:\Users\USER\AppData\Local\Temp\ipykernel_28892\1237133930.py:40: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  orders = pd.read_excel(file_path, sheet_name="Orders")
C:\Users\USER\AppData\Local\Temp\ipykernel_28892\1237133930.py:41: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  returns = pd.read_excel(file_path, sheet_name="Returns")
C:\Users\USER\AppData\Local\Temp\ipykernel_28892\1237133930.py:42: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  people = pd.read_excel(file_path, sheet_name="People")



DATASET DIMENSIONS
Orders: (10194, 21)
Returns: (800, 2)
People: (4, 2)

ORDERS COLUMNS:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State/Province,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2021-103800,2021-01-03,2021-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.200,5.551
1,2,US-2021-112326,2021-01-04,2021-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.800,-5.487
2,3,US-2021-112326,2021-01-04,2021-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.200,4.272
3,4,US-2021-112326,2021-01-04,2021-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.200,-64.775
4,5,US-2021-141817,2021-01-05,2021-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.200,4.884


### 2. Target Definition

Two prediction targets are established:

- **Regression target:** Profit, representing the numerical financial outcome of each transaction.
- **Classification target:** Loss_Making, a binary variable indicating whether a transaction generates a negative profit.

The classification target is defined as 1 for loss-making transactions and 0 for profitable or break-even transactions.

In [2]:

# ============================================
# STEP 2: DEFINE PREDICTION TARGETS
# ============================================

# Create a separate working dataset
orders_ml = orders.copy()

# Regression target
y_regression = orders_ml["Profit"].copy()

# Classification target
orders_ml["Loss_Making"] = (
    orders_ml["Profit"] < 0
).astype(int)

y_classification = orders_ml["Loss_Making"].copy()

# Examine target distributions
print("REGRESSION TARGET")
print("Target: Profit")
print(y_regression.describe())

print("\nCLASSIFICATION TARGET")
print("Target: Loss_Making")

class_counts = y_classification.value_counts().sort_index()

print("\nClass counts:")
print(class_counts.rename(index={
    0: "Not Loss-Making",
    1: "Loss-Making"
}))

print("\nClass percentages:")
print(
    (y_classification.value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

# Verify target integrity
print("\nTARGET VERIFICATION")
print("Missing Profit values:", y_regression.isna().sum())
print("Missing classification values:",
      y_classification.isna().sum())

print("Break-even transactions:",
      (y_regression == 0).sum())

print("\nUnique orders:", orders_ml["Order ID"].nunique())

print("Transactions:", len(orders_ml))

print("Transactions per order (average):",
      round(
          len(orders_ml) / orders_ml["Order ID"].nunique(), 2
      ))


REGRESSION TARGET
Target: Profit
count   10,194.000
mean        28.673
std        232.465
min     -6,599.978
25%          1.761
50%          8.690
75%         29.298
max      8,399.976
Name: Profit, dtype: float64

CLASSIFICATION TARGET
Target: Loss_Making

Class counts:
Loss_Making
Not Loss-Making    8293
Loss-Making        1901
Name: count, dtype: int64

Class percentages:
Loss_Making
0   81.350
1   18.650
Name: proportion, dtype: float64

TARGET VERIFICATION
Missing Profit values: 0
Missing classification values: 0
Break-even transactions: 65

Unique orders: 5111
Transactions: 10194
Transactions per order (average): 1.99
